# Metadata Summary: Study Cohort Characterization

Summarize the full cohort including epilepsy (DR/DS), PNEE, one seizure only, and syncope.  
Check data availability across modalities and compare clinical features between groups.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact
import warnings
warnings.filterwarnings('ignore')

meta = pd.read_csv('../data/metadata_drug_responsiveness.csv')
meta = meta.rename(columns={'record_id': 'SubjectID'})

# Recode: merge 'mixed focal/gen' into 'unclassified'
meta['group_1'] = meta['group_1'].replace('mixed focal/gen', 'unclassified')

print(f'Total subjects in metadata: {len(meta)}')
print(f'\nBy group_1:')
print(meta['group_1'].value_counts().to_string())
print(f'\nBy group_2:')
print(meta['group_2'].value_counts().to_string())

Total subjects in metadata: 450

By group_1:
group_1
focal           267
generalized      46
pnee             37
unclassified     36
syncope          33
one_sz_only      31

By group_2:
group_2
drug-responsive    207
drug-resistant     142
pnee                37
syncope             33
one_sz_only         31


## 1. All Subjects in Metadata

In [2]:
# Full cross-tabulation: group_1 x group_2
ct = pd.crosstab(meta['group_1'], meta['group_2'], margins=True)
ct.index.name = 'Epilepsy Type'
ct.columns.name = ''
display(ct)

,drug-resistant,drug-responsive,one_sz_only,pnee,syncope,All
Epilepsy Type,,,,,,
focal,113,154,0,0,0,267
generalized,20,26,0,0,0,46
one_sz_only,0,0,31,0,0,31
pnee,0,0,0,37,0,37
syncope,0,0,0,0,33,33
unclassified,9,27,0,0,0,36
All,142,207,31,37,33,450


## 2. Subjects with Both Morphometry and Tractometry Data

In [3]:
# Morphometry subjects: need both CT and SV
ct_m = pd.read_csv('../data/centilebrain_zscores/zscore_CorticalThickness_male.csv')
ct_f = pd.read_csv('../data/centilebrain_zscores/zscore_CorticalThickness_female.csv')
sv_m = pd.read_csv('../data/centilebrain_zscores/zscore_SubcorticalVolume_male.csv')
sv_f = pd.read_csv('../data/centilebrain_zscores/zscore_SubcorticalVolume_female.csv')

ct_subs = set(ct_m['SubjectID']) | set(ct_f['SubjectID'])
sv_subs = set(sv_m['SubjectID']) | set(sv_f['SubjectID'])
morph_subs = ct_subs & sv_subs

# Tractometry subjects
tract_raw = pd.read_csv('../data/node_zscores.csv', usecols=['subject', 'cohort'])
tract_subs = set(tract_raw[tract_raw['cohort'] == 'Clinical']['subject'].unique())

print(f'Unique morphometry subjects (CT + SV): {len(morph_subs)}')
print(f'Unique tractometry subjects (clinical only): {len(tract_subs)}')

both_subs = morph_subs & tract_subs
print(f'Subjects with both modalities: {len(both_subs)}')

# Filter metadata to shared cohort
cohort = meta[meta['SubjectID'].isin(both_subs)].copy()
cohort = cohort.drop_duplicates(subset='SubjectID', keep='first')  # one subject duplicated in metadata,
# same fix applied throughout the AUC/tractometry notebooks -- keeps Table 1's N consistent with the
# cohort actually used in the results figures/tables.
print(f'\nFinalized shared cohort: {len(cohort)}')
print(f'\nBy group_1:')
print(cohort['group_1'].value_counts().to_string())
print(f'\nBy group_2:')
print(cohort['group_2'].value_counts().to_string())

# Overlap breakdown
morph_only = morph_subs - tract_subs
tract_only = tract_subs - morph_subs
print(f'\nMorphometry only (no tractometry): {len(morph_only)}')
print(f'Tractometry only (no morphometry): {len(tract_only)}')

Unique morphometry subjects (CT + SV): 355
Unique tractometry subjects (clinical only): 360
Subjects with both modalities: 355

Finalized shared cohort: 355

By group_1:
group_1
focal           203
generalized      37
unclassified     33
pnee             32
syncope          25
one_sz_only      25

By group_2:
group_2
drug-responsive    176
drug-resistant      97
pnee                32
syncope             25
one_sz_only         25

Morphometry only (no tractometry): 0
Tractometry only (no morphometry): 5


## 3. Finalized Cohort Breakdown

In [4]:
# Cross-tabulation of finalized cohort
ct = pd.crosstab(cohort['group_1'], cohort['group_2'], margins=True)
ct.index.name = 'Type'
ct.columns.name = ''
display(ct)

# Define subsets
epi = cohort[cohort['group_2'].isin(['drug-resistant', 'drug-responsive'])].copy()
pnee = cohort[cohort['group_1'] == 'pnee'].copy()
one_sz = cohort[cohort['group_1'] == 'one_sz_only'].copy()
syncope = cohort[cohort['group_1'] == 'syncope'].copy()

print(f'\nEpilepsy (DR+DS): {len(epi)}')
print(f'  Drug-responsive: {(epi["group_2"] == "drug-responsive").sum()}')
print(f'  Drug-resistant: {(epi["group_2"] == "drug-resistant").sum()}')
print(f'PNEE: {len(pnee)}')
print(f'One seizure only: {len(one_sz)}')
print(f'Syncope: {len(syncope)}')

# Epilepsy subtype breakdown
print(f'\nEpilepsy subtype breakdown:')
for etype in ['focal', 'generalized', 'unclassified']:
    sub = epi[epi['group_1'] == etype]
    n_dr = (sub['group_2'] == 'drug-resistant').sum()
    n_ds = (sub['group_2'] == 'drug-responsive').sum()
    print(f'  {etype.title()}: {len(sub)} (DR={n_dr}, DS={n_ds})')

,drug-resistant,drug-responsive,one_sz_only,pnee,syncope,All
Type,,,,,,
focal,73,130,0,0,0,203
generalized,17,20,0,0,0,37
one_sz_only,0,0,25,0,0,25
pnee,0,0,0,32,0,32
syncope,0,0,0,0,25,25
unclassified,7,26,0,0,0,33
All,97,176,25,32,25,355



Epilepsy (DR+DS): 273
  Drug-responsive: 176
  Drug-resistant: 97
PNEE: 32
One seizure only: 25
Syncope: 25

Epilepsy subtype breakdown:
  Focal: 203 (DR=73, DS=130)
  Generalized: 37 (DR=17, DS=20)
  Unclassified: 33 (DR=7, DS=26)


## 4. Demographics: All Groups

Summary of sex, age at MRI, and age of onset across all groups.

In [5]:
# Demographics summary across all groups
groups = {
    'Epilepsy DS': epi[epi['group_2'] == 'drug-responsive'],
    'Epilepsy DR': epi[epi['group_2'] == 'drug-resistant'],
    'PNEE': pnee,
    'One Sz Only': one_sz,
    'Syncope': syncope
}

demo_rows = []
for name, data in groups.items():
    sex_vals = data['sex'].dropna()
    age_mri = pd.to_numeric(data['age_MRI'], errors='coerce').dropna()
    age_onset = pd.to_numeric(data['age_onset'], errors='coerce').dropna()
    n_female = (sex_vals == 'F').sum()
    n_male = (sex_vals == 'M').sum()
    demo_rows.append({
        'Group': name,
        'n': len(data),
        'Female / Male': f'{n_female} / {n_male}',
        '% Female': f'{100 * n_female / len(sex_vals):.1f}%' if len(sex_vals) > 0 else 'N/A',
        'Age at MRI (median [IQR])': f'{age_mri.median():.1f} [{age_mri.quantile(0.25):.1f}-{age_mri.quantile(0.75):.1f}]' if len(age_mri) > 0 else 'N/A',
        'Age of Onset (median [IQR])': f'{age_onset.median():.1f} [{age_onset.quantile(0.25):.1f}-{age_onset.quantile(0.75):.1f}]' if len(age_onset) > 0 else 'N/A'
    })

demo_df = pd.DataFrame(demo_rows)
display(demo_df)

,Group,n,Female / Male,% Female,Age at MRI (median [IQR]),Age of Onset (median [IQR])
0,Epilepsy DS,176,82 / 94,46.6%,35.5 [27.0-48.8],26.5 [18.0-41.8]
1,Epilepsy DR,97,45 / 52,46.4%,36.0 [26.0-45.0],19.0 [10.0-31.2]
2,PNEE,32,25 / 7,78.1%,35.5 [26.9-42.7],27.5 [21.2-39.0]
3,One Sz Only,25,14 / 11,56.0%,41.0 [27.0-47.0],39.0 [25.0-47.0]
4,Syncope,25,17 / 8,68.0%,33.8 [29.0-49.5],33.0 [23.0-46.0]


## 5. Epilepsy vs PNEE: Clinical Comparison

Compare demographics between epilepsy (all DR+DS) and PNEE.

In [6]:
def compare_two_groups(group_a, group_b, label_a, label_b):
    """Compare demographics and clinical features between two groups."""
    results = []

    def cohens_d(x, y):
        nx, ny = len(x), len(y)
        pooled_std = np.sqrt(((nx - 1) * x.std()**2 + (ny - 1) * y.std()**2) / (nx + ny - 2))
        return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else 0.0

    # Sex
    a_f = (group_a['sex'] == 'F').sum()
    a_m = (group_a['sex'] == 'M').sum()
    b_f = (group_b['sex'] == 'F').sum()
    b_m = (group_b['sex'] == 'M').sum()
    table = [[a_f, a_m], [b_f, b_m]]
    _, p = fisher_exact(table)
    chi2, _, _, _ = chi2_contingency(table, correction=False)
    results.append({
        'Variable': 'Sex (F/M)',
        f'{label_a} (n={len(group_a)})': f'{a_f}/{a_m}',
        f'{label_b} (n={len(group_b)})': f'{b_f}/{b_m}',
        'Test': 'Fisher exact',
        'Effect Size': f'\u03c7\u00b2={chi2:.2f}',
        'p-value': p
    })

    # Continuous variables
    cont_vars = [('age_MRI', 'Age at MRI'), ('age_onset', 'Age of Onset')]
    for var, label in cont_vars:
        a_vals = pd.to_numeric(group_a[var], errors='coerce').dropna()
        b_vals = pd.to_numeric(group_b[var], errors='coerce').dropna()
        if len(a_vals) > 0 and len(b_vals) > 0:
            stat, p = mannwhitneyu(a_vals, b_vals, alternative='two-sided')
            d = cohens_d(a_vals, b_vals)
            results.append({
                'Variable': label,
                f'{label_a} (n={len(group_a)})': f'{a_vals.median():.1f} [{a_vals.quantile(0.25):.1f}-{a_vals.quantile(0.75):.1f}]',
                f'{label_b} (n={len(group_b)})': f'{b_vals.median():.1f} [{b_vals.quantile(0.25):.1f}-{b_vals.quantile(0.75):.1f}]',
                'Test': 'Mann-Whitney U',
                'Effect Size': f'd={d:.2f}',
                'p-value': p
            })

    res_df = pd.DataFrame(results)
    res_df['Sig'] = res_df['p-value'].apply(lambda p: '*' if p < 0.05 else '')
    res_df['p-value'] = res_df['p-value'].apply(lambda p: f'{p:.4f}')
    return res_df

print('=== Epilepsy vs PNEE ===')
epi_vs_pnee_demo = compare_two_groups(epi, pnee, 'Epilepsy', 'PNEE')
display(epi_vs_pnee_demo)

=== Epilepsy vs PNEE ===


,Variable,Epilepsy (n=273),PNEE (n=32),Test,Effect Size,p-value,Sig
0,Sex (F/M),127/146,25/7,Fisher exact,χ²=11.44,0.0007,*
1,Age at MRI,36.0 [26.0-46.0],35.5 [26.9-42.7],Mann-Whitney U,d=0.22,0.4903,
2,Age of Onset,24.0 [13.2-38.0],27.5 [21.2-39.0],Mann-Whitney U,d=-0.13,0.2015,


## 6. Epilepsy vs One Seizure Only: Clinical Comparison

In [7]:
print('=== Epilepsy vs One Seizure Only ===')
epi_vs_onesz_demo = compare_two_groups(epi, one_sz, 'Epilepsy', 'One Sz Only')
display(epi_vs_onesz_demo)

=== Epilepsy vs One Seizure Only ===


,Variable,Epilepsy (n=273),One Sz Only (n=25),Test,Effect Size,p-value,Sig
0,Sex (F/M),127/146,14/11,Fisher exact,χ²=0.83,0.4070,
1,Age at MRI,36.0 [26.0-46.0],41.0 [27.0-47.0],Mann-Whitney U,d=-0.07,0.4970,
2,Age of Onset,24.0 [13.2-38.0],39.0 [25.0-47.0],Mann-Whitney U,d=-0.62,0.0009,*


## 7. Drug-Responsive vs Drug-Resistant: Clinical Comparison

Compare sex, age_MRI, age_onset, epi_duration, convulsive_sz, status_epilepticus, and mri_lesion between DR and DS.

In [8]:
ds = epi[epi['group_2'] == 'drug-responsive'].copy()
dr = epi[epi['group_2'] == 'drug-resistant'].copy()

print(f'Drug-Responsive (DS): n={len(ds)}')
print(f'Drug-Resistant (DR): n={len(dr)}')
print()

results = []

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx - 1) * x.std()**2 + (ny - 1) * y.std()**2) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else 0.0

# --- Sex ---
ds_female = (ds['sex'] == 'F').sum()
ds_male = (ds['sex'] == 'M').sum()
dr_female = (dr['sex'] == 'F').sum()
dr_male = (dr['sex'] == 'M').sum()
table = [[ds_female, ds_male], [dr_female, dr_male]]
odds_ratio, p = fisher_exact(table)
chi2, _, _, _ = chi2_contingency(table, correction=False)
results.append({
    'Variable': 'Sex (F/M)',
    'DS (n)': len(ds),
    'DS': f'{ds_female}/{ds_male}',
    'DR (n)': len(dr),
    'DR': f'{dr_female}/{dr_male}',
    'Test': 'Fisher exact',
    'Effect Size': f'\u03c7\u00b2={chi2:.2f}',
    'p-value': p
})
print(f'Sex (F/M):')
print(f'  DS: {ds_female}F / {ds_male}M')
print(f'  DR: {dr_female}F / {dr_male}M')
print(f'  \u03c7\u00b2 = {chi2:.2f}, p = {p:.4f}{"  *" if p < 0.05 else ""}\n')

# --- Continuous variables ---
for var, label in [('age_MRI', 'Age at MRI'),
                   ('age_onset', 'Age of Onset'),
                   ('epi_duration', 'Epilepsy Duration (yrs)')]:
    ds_vals = pd.to_numeric(ds[var], errors='coerce').dropna()
    dr_vals = pd.to_numeric(dr[var], errors='coerce').dropna()
    stat, p = mannwhitneyu(ds_vals, dr_vals, alternative='two-sided')
    d = cohens_d(ds_vals, dr_vals)
    results.append({
        'Variable': label,
        'DS (n)': len(ds_vals),
        'DS': f'{ds_vals.median():.1f} ({ds_vals.quantile(0.25):.1f}-{ds_vals.quantile(0.75):.1f})',
        'DR (n)': len(dr_vals),
        'DR': f'{dr_vals.median():.1f} ({dr_vals.quantile(0.25):.1f}-{dr_vals.quantile(0.75):.1f})',
        'Test': 'Mann-Whitney U',
        'Effect Size': f'd={d:.2f}',
        'p-value': p
    })
    print(f'{label}:')
    print(f'  DS: median={ds_vals.median():.1f}, IQR=[{ds_vals.quantile(0.25):.1f}-{ds_vals.quantile(0.75):.1f}], n={len(ds_vals)}')
    print(f'  DR: median={dr_vals.median():.1f}, IQR=[{dr_vals.quantile(0.25):.1f}-{dr_vals.quantile(0.75):.1f}], n={len(dr_vals)}')
    print(f'  d = {d:.2f}, p = {p:.4f}{"  *" if p < 0.05 else ""}\n')

# --- Binary variables ---
for var, label in [('convulsive_sz', 'History of Convulsions'),
                   ('status_epilepticus', 'History of Status Epilepticus'),
                   ('mri_lesion', 'MRI Lesion')]:
    ds_vals = pd.to_numeric(ds[var], errors='coerce').dropna()
    dr_vals = pd.to_numeric(dr[var], errors='coerce').dropna()
    a = int(ds_vals.sum())
    b = int(len(ds_vals) - a)
    c = int(dr_vals.sum())
    d_no = int(len(dr_vals) - c)
    table = [[a, b], [c, d_no]]
    odds_ratio, p = fisher_exact(table)
    chi2, _, _, _ = chi2_contingency(table, correction=False)
    ds_pct = 100 * a / len(ds_vals) if len(ds_vals) > 0 else 0
    dr_pct = 100 * c / len(dr_vals) if len(dr_vals) > 0 else 0
    results.append({
        'Variable': label,
        'DS (n)': len(ds_vals),
        'DS': f'{a}/{len(ds_vals)} ({ds_pct:.1f}%)',
        'DR (n)': len(dr_vals),
        'DR': f'{c}/{len(dr_vals)} ({dr_pct:.1f}%)',
        'Test': 'Fisher exact',
        'Effect Size': f'\u03c7\u00b2={chi2:.2f}',
        'p-value': p
    })
    print(f'{label}:')
    print(f'  DS: {a}/{len(ds_vals)} ({ds_pct:.1f}%)')
    print(f'  DR: {c}/{len(dr_vals)} ({dr_pct:.1f}%)')
    print(f'  \u03c7\u00b2 = {chi2:.2f}, p = {p:.4f}{"  *" if p < 0.05 else ""}\n')

summary = pd.DataFrame(results)
summary['Sig'] = summary['p-value'].apply(lambda p: '*' if p < 0.05 else '')
summary['p-value'] = summary['p-value'].apply(lambda p: f'{p:.4f}')
print('\n--- Summary Table: DR vs DS ---')
display(summary)

Drug-Responsive (DS): n=176
Drug-Resistant (DR): n=97

Sex (F/M):
  DS: 82F / 94M
  DR: 45F / 52M
  χ² = 0.00, p = 1.0000

Age at MRI:
  DS: median=35.5, IQR=[27.0-48.8], n=176
  DR: median=36.0, IQR=[26.0-45.0], n=97
  d = 0.17, p = 0.4371

Age of Onset:
  DS: median=26.5, IQR=[18.0-41.8], n=174
  DR: median=19.0, IQR=[10.0-31.2], n=96
  d = 0.50, p = 0.0002  *

Epilepsy Duration (yrs):
  DS: median=5.0, IQR=[1.0-13.0], n=175
  DR: median=10.0, IQR=[6.0-23.0], n=97
  d = -0.51, p = 0.0000  *

History of Convulsions:
  DS: 136/176 (77.3%)
  DR: 85/97 (87.6%)
  χ² = 4.35, p = 0.0378  *

History of Status Epilepticus:
  DS: 7/176 (4.0%)
  DR: 8/97 (8.2%)
  χ² = 2.20, p = 0.1676

MRI Lesion:
  DS: 58/176 (33.0%)
  DR: 31/97 (32.0%)
  χ² = 0.03, p = 0.8935


--- Summary Table: DR vs DS ---


,Variable,DS (n),DS,DR (n),DR,Test,Effect Size,p-value,Sig
0,Sex (F/M),176,82/94,97,45/52,Fisher exact,χ²=0.00,1.0000,
1,Age at MRI,176,35.5 (27.0-48.8),97,36.0 (26.0-45.0),Mann-Whitney U,d=0.17,0.4371,
2,Age of Onset,174,26.5 (18.0-41.8),96,19.0 (10.0-31.2),Mann-Whitney U,d=0.50,0.0002,*
3,Epilepsy Duration (yrs),175,5.0 (1.0-13.0),97,10.0 (6.0-23.0),Mann-Whitney U,d=-0.51,0.0000,*
4,History of Convulsions,176,136/176 (77.3%),97,85/97 (87.6%),Fisher exact,χ²=4.35,0.0378,*
5,History of Status Epilepticus,176,7/176 (4.0%),97,8/97 (8.2%),Fisher exact,χ²=2.20,0.1676,
6,MRI Lesion,176,58/176 (33.0%),97,31/97 (32.0%),Fisher exact,χ²=0.03,0.8935,


## 8. DR vs DS by Epilepsy Subtype

Compare DR vs DS separately within focal, generalized, and unclassified epilepsy.

In [9]:
for etype in ['focal', 'generalized', 'unclassified']:
    sub = epi[epi['group_1'] == etype]
    sub_ds = sub[sub['group_2'] == 'drug-responsive']
    sub_dr = sub[sub['group_2'] == 'drug-resistant']
    if len(sub_ds) < 5 or len(sub_dr) < 5:
        print(f'\n=== {etype.title()} ===')
        print(f'  Skipping: DS={len(sub_ds)}, DR={len(sub_dr)} (too few)')
        continue
    print(f'\n=== {etype.title()} (DS={len(sub_ds)}, DR={len(sub_dr)}) ===')
    res = compare_two_groups(sub_dr, sub_ds, 'DR', 'DS')
    display(res)


=== Focal (DS=130, DR=73) ===


,Variable,DR (n=73),DS (n=130),Test,Effect Size,p-value,Sig
0,Sex (F/M),33/40,60/70,Fisher exact,χ²=0.02,1.0000,
1,Age at MRI,38.0 [27.0-48.0],37.5 [29.0-53.8],Mann-Whitney U,d=-0.14,0.6802,
2,Age of Onset,24.0 [11.0-35.0],29.5 [19.0-45.8],Mann-Whitney U,d=-0.48,0.0043,*



=== Generalized (DS=20, DR=17) ===


,Variable,DR (n=17),DS (n=20),Test,Effect Size,p-value,Sig
0,Sex (F/M),9/8,11/9,Fisher exact,χ²=0.02,1.0000,
1,Age at MRI,24.0 [20.0-29.0],24.5 [20.8-29.0],Mann-Whitney U,d=-0.06,0.8542,
2,Age of Onset,12.0 [9.0-16.0],12.0 [8.0-20.2],Mann-Whitney U,d=-0.44,0.4825,



=== Unclassified (DS=26, DR=7) ===


,Variable,DR (n=7),DS (n=26),Test,Effect Size,p-value,Sig
0,Sex (F/M),3/4,11/15,Fisher exact,χ²=0.00,1.0000,
1,Age at MRI,33.0 [32.5-34.5],34.5 [29.0-46.0],Mann-Whitney U,d=-0.36,0.6913,
2,Age of Onset,11.5 [5.4-12.8],25.0 [17.5-32.0],Mann-Whitney U,d=-1.06,0.0196,*


## 10. Focal vs. Generalized: Clinical Comparison
Compare sex, age_MRI, age_onset, epi_duration, convulsive_sz, status_epilepticus, and mri_lesion between focal and generalized epilepsy.

In [10]:
focal = epi[epi['group_1'] == 'focal'].copy()
generalized = epi[epi['group_1'] == 'generalized'].copy()

print(f'Focal: n={len(focal)}')
print(f'Generalized: n={len(generalized)}')
print()

results = []

def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx - 1) * x.std()**2 + (ny - 1) * y.std()**2) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else 0.0

# --- Sex ---
f_female = (focal['sex'] == 'F').sum()
f_male = (focal['sex'] == 'M').sum()
g_female = (generalized['sex'] == 'F').sum()
g_male = (generalized['sex'] == 'M').sum()
table = [[f_female, f_male], [g_female, g_male]]
odds_ratio, p = fisher_exact(table)
chi2, _, _, _ = chi2_contingency(table, correction=False)
results.append({
    'Variable': 'Sex (F/M)',
    'Focal (n)': len(focal),
    'Focal': f'{f_female}/{f_male}',
    'Generalized (n)': len(generalized),
    'Generalized': f'{g_female}/{g_male}',
    'Test': 'Fisher exact',
    'Effect Size': f'\u03c7\u00b2={chi2:.2f}',
    'p-value': p
})
print(f'Sex (F/M):')
print(f'  Focal: {f_female}F / {f_male}M')
print(f'  Generalized: {g_female}F / {g_male}M')
print(f'  \u03c7\u00b2 = {chi2:.2f}, p = {p:.4f}{"  *" if p < 0.05 else ""}\n')

# --- Continuous variables ---
for var, label in [('age_MRI', 'Age at MRI'),
                   ('age_onset', 'Age of Onset'),
                   ('epi_duration', 'Epilepsy Duration (yrs)')]:
    f_vals = pd.to_numeric(focal[var], errors='coerce').dropna()
    g_vals = pd.to_numeric(generalized[var], errors='coerce').dropna()
    stat, p = mannwhitneyu(f_vals, g_vals, alternative='two-sided')
    d = cohens_d(f_vals, g_vals)
    results.append({
        'Variable': label,
        'Focal (n)': len(f_vals),
        'Focal': f'{f_vals.median():.1f} ({f_vals.quantile(0.25):.1f}-{f_vals.quantile(0.75):.1f})',
        'Generalized (n)': len(g_vals),
        'Generalized': f'{g_vals.median():.1f} ({g_vals.quantile(0.25):.1f}-{g_vals.quantile(0.75):.1f})',
        'Test': 'Mann-Whitney U',
        'Effect Size': f'd={d:.2f}',
        'p-value': p
    })
    print(f'{label}:')
    print(f'  Focal: median={f_vals.median():.1f}, IQR=[{f_vals.quantile(0.25):.1f}-{f_vals.quantile(0.75):.1f}], n={len(f_vals)}')
    print(f'  Generalized: median={g_vals.median():.1f}, IQR=[{g_vals.quantile(0.25):.1f}-{g_vals.quantile(0.75):.1f}], n={len(g_vals)}')
    print(f'  d = {d:.2f}, p = {p:.4f}{"  *" if p < 0.05 else ""}\n')

# --- Binary variables ---
for var, label in [('convulsive_sz', 'History of Convulsions'),
                   ('status_epilepticus', 'History of Status Epilepticus'),
                   ('mri_lesion', 'MRI Lesion')]:
    f_vals = pd.to_numeric(focal[var], errors='coerce').dropna()
    g_vals = pd.to_numeric(generalized[var], errors='coerce').dropna()
    a = int(f_vals.sum())
    b = int(len(f_vals) - a)
    c = int(g_vals.sum())
    d_no = int(len(g_vals) - c)
    table = [[a, b], [c, d_no]]
    odds_ratio, p = fisher_exact(table)
    chi2, _, _, _ = chi2_contingency(table, correction=False)
    f_pct = 100 * a / len(f_vals) if len(f_vals) > 0 else 0
    g_pct = 100 * c / len(g_vals) if len(g_vals) > 0 else 0
    results.append({
        'Variable': label,
        'Focal (n)': len(f_vals),
        'Focal': f'{a}/{len(f_vals)} ({f_pct:.1f}%)',
        'Generalized (n)': len(g_vals),
        'Generalized': f'{c}/{len(g_vals)} ({g_pct:.1f}%)',
        'Test': 'Fisher exact',
        'Effect Size': f'\u03c7\u00b2={chi2:.2f}',
        'p-value': p
    })
    print(f'{label}:')
    print(f'  Focal: {a}/{len(f_vals)} ({f_pct:.1f}%)')
    print(f'  Generalized: {c}/{len(g_vals)} ({g_pct:.1f}%)')
    print(f'  \u03c7\u00b2 = {chi2:.2f}, p = {p:.4f}{"  *" if p < 0.05 else ""}\n')

summary_fvg = pd.DataFrame(results)
summary_fvg['Sig'] = summary_fvg['p-value'].apply(lambda p: '*' if p < 0.05 else '')
summary_fvg['p-value'] = summary_fvg['p-value'].apply(lambda p: f'{p:.4f}')
print('\n--- Summary Table: Focal vs Generalized ---')
display(summary_fvg)

Focal: n=203
Generalized: n=37

Sex (F/M):
  Focal: 93F / 110M
  Generalized: 20F / 17M
  χ² = 0.85, p = 0.3758

Age at MRI:
  Focal: median=38.0, IQR=[29.0-51.0], n=203
  Generalized: median=24.0, IQR=[20.0-29.0], n=37
  d = 0.85, p = 0.0000  *

Age of Onset:
  Focal: median=28.0, IQR=[18.0-41.0], n=203
  Generalized: median=12.0, IQR=[8.0-18.0], n=37
  d = 0.96, p = 0.0000  *

Epilepsy Duration (yrs):
  Focal: median=6.0, IQR=[2.0-15.0], n=203
  Generalized: median=12.0, IQR=[5.0-19.0], n=37
  d = -0.29, p = 0.0557

History of Convulsions:
  Focal: 155/203 (76.4%)
  Generalized: 35/37 (94.6%)
  χ² = 6.31, p = 0.0137  *

History of Status Epilepticus:
  Focal: 11/203 (5.4%)
  Generalized: 3/37 (8.1%)
  χ² = 0.41, p = 0.4585

MRI Lesion:
  Focal: 79/203 (38.9%)
  Generalized: 1/37 (2.7%)
  χ² = 18.47, p = 0.0000  *


--- Summary Table: Focal vs Generalized ---


,Variable,Focal (n),Focal,Generalized (n),Generalized,Test,Effect Size,p-value,Sig
0,Sex (F/M),203,93/110,37,20/17,Fisher exact,χ²=0.85,0.3758,
1,Age at MRI,203,38.0 (29.0-51.0),37,24.0 (20.0-29.0),Mann-Whitney U,d=0.85,0.0000,*
2,Age of Onset,203,28.0 (18.0-41.0),37,12.0 (8.0-18.0),Mann-Whitney U,d=0.96,0.0000,*
3,Epilepsy Duration (yrs),203,6.0 (2.0-15.0),37,12.0 (5.0-19.0),Mann-Whitney U,d=-0.29,0.0557,
4,History of Convulsions,203,155/203 (76.4%),37,35/37 (94.6%),Fisher exact,χ²=6.31,0.0137,*
5,History of Status Epilepticus,203,11/203 (5.4%),37,3/37 (8.1%),Fisher exact,χ²=0.41,0.4585,
6,MRI Lesion,203,79/203 (38.9%),37,1/37 (2.7%),Fisher exact,χ²=18.47,0.0000,*


## 9. Composite Table 1: Clinical Characteristics Across All Comparisons

In [11]:
# ── Composite Table 1: Clinical characteristics across all comparisons ──────
focal_epi = epi[epi['group_1'] == 'focal'].copy()
gen_epi   = epi[epi['group_1'] == 'generalized'].copy()
ds        = epi[epi['group_2'] == 'drug-responsive'].copy()
dr        = epi[epi['group_2'] == 'drug-resistant'].copy()

# ── Formatting helpers ───────────────────────────────────────────────────────
def fmt_sex(data):
    f = (data['sex'] == 'F').sum()
    m = (data['sex'] == 'M').sum()
    return f'{f}/{m}'

def fmt_median_iqr(series, decimals=1):
    vals = pd.to_numeric(series, errors='coerce').dropna()
    if len(vals) == 0:
        return 'N/A'
    fmt = f'{{:.{decimals}f}}'
    return f'{fmt.format(vals.median())} ({fmt.format(vals.quantile(0.25))}-{fmt.format(vals.quantile(0.75))})'

def fmt_n_pct(series):
    vals = pd.to_numeric(series, errors='coerce').dropna()
    if len(vals) == 0:
        return 'N/A'
    n   = int((vals == 1).sum())
    pct = 100 * n / len(vals)
    return f'{n} ({pct:.0f}%)'

age_cols = {'age_onset', 'age_MRI', 'epi_duration'}

def fmt_cell(data, var_col):
    if var_col == 'sex':
        return fmt_sex(data)
    elif var_col in ('convulsive_sz', 'status_epilepticus', 'mri_lesion'):
        return fmt_n_pct(data[var_col])
    elif var_col in age_cols:
        return fmt_median_iqr(data[var_col], decimals=0)
    else:
        return fmt_median_iqr(data[var_col])

def compute_p(group_a, group_b, var_col):
    if var_col == 'sex':
        a_f = (group_a['sex'] == 'F').sum(); a_m = (group_a['sex'] == 'M').sum()
        b_f = (group_b['sex'] == 'F').sum(); b_m = (group_b['sex'] == 'M').sum()
        _, p = fisher_exact([[a_f, a_m], [b_f, b_m]])
        return p
    elif var_col in ('convulsive_sz', 'status_epilepticus', 'mri_lesion'):
        a = pd.to_numeric(group_a[var_col], errors='coerce').dropna()
        b = pd.to_numeric(group_b[var_col], errors='coerce').dropna()
        if len(a) == 0 or len(b) == 0: return None
        _, p = fisher_exact([[int((a==1).sum()), int((a==0).sum())],
                             [int((b==1).sum()), int((b==0).sum())]])
        return p
    else:
        a = pd.to_numeric(group_a[var_col], errors='coerce').dropna()
        b = pd.to_numeric(group_b[var_col], errors='coerce').dropna()
        if len(a) == 0 or len(b) == 0: return None
        _, p = mannwhitneyu(a, b, alternative='two-sided')
        return p

def fmt_p(p):
    if p is None: return 'N/A'
    return '<0.001' if p < 0.001 else f'{p:.3f}'

# ── Groups ───────────────────────────────────────────────────────────────────
comparisons = [
    ('Comparison 1', epi,       pnee,    'Epilepsy',        'Functional Seizures Only'),
    ('Comparison 2', focal_epi, gen_epi, 'Focal',           'Generalized'),
    ('Comparison 3', ds,        dr,      'Drug-responsive',  'Drug-resistant'),
]

# Variables where Functional Seizures Only column is N/A
pnee_na = {'age_onset', 'epi_duration', 'convulsive_sz', 'status_epilepticus', 'mri_lesion'}

variables = [
    ('Sex at birth, female/male',                 'sex'),
    ('Age of epilepsy onset, median (IQR)',        'age_onset'),
    ('Age at MRI scan, median (IQR)',              'age_MRI'),
    ('Epilepsy duration (years), median (IQR)',    'epi_duration'),
    ('History of convulsive seizures, n (%)',      'convulsive_sz'),
    ('History of status epilepticus, n (%)',       'status_epilepticus'),
    ('Epileptogenic lesion on MRI, n (%)',         'mri_lesion'),
]

# ── Build column index ────────────────────────────────────────────────────────
col_tuples = []
for comp_label, ga, gb, la, lb in comparisons:
    col_tuples += [
        (comp_label, f'{la}\n(n={len(ga)})'),
        (comp_label, f'{lb}\n(n={len(gb)})'),
        (comp_label, 'p'),
    ]

# ── Build rows ────────────────────────────────────────────────────────────────
rows = {}
for var_label, var_col in variables:
    row = {}
    for comp_label, ga, gb, la, lb in comparisons:
        is_pnee_comp = (comp_label == 'Comparison 1')
        col_a = (comp_label, f'{la}\n(n={len(ga)})')
        col_b = (comp_label, f'{lb}\n(n={len(gb)})')
        col_p = (comp_label, 'p')

        row[col_a] = fmt_cell(ga, var_col)
        row[col_b] = 'N/A' if (is_pnee_comp and var_col in pnee_na) else fmt_cell(gb, var_col)
        row[col_p] = 'N/A' if (is_pnee_comp and var_col in pnee_na) else fmt_p(compute_p(ga, gb, var_col))

    rows[var_label] = row

# ── Assemble and display ──────────────────────────────────────────────────────
col_idx = pd.MultiIndex.from_tuples(col_tuples, names=['', ''])
table1  = pd.DataFrame(rows, index=col_idx).T
table1.index.name = ''

display(table1)

# ── Save ──────────────────────────────────────────────────────────────────────
import os
os.makedirs('../outputs', exist_ok=True)
table1.to_csv('../outputs/table1_clinical_characteristics.csv')
print('Saved: ../outputs/table1_clinical_characteristics.csv')


Comparison 1  \
                                        Epilepsy\n(n=273)   
                                                            
Sex at birth, female/male                         127/146   
Age of epilepsy onset, median (IQR)            24 (13-38)   
Age at MRI scan, median (IQR)                  36 (26-46)   
Epilepsy duration (years), median (IQR)          8 (2-17)   
History of convulsive seizures, n (%)           221 (81%)   
History of status epilepticus, n (%)              15 (5%)   
Epileptogenic lesion on MRI, n (%)               89 (33%)   

                                                                          \
                                        Functional Seizures Only\n(n=32)   
                                                                           
Sex at birth, female/male                                           25/7   
Age of epilepsy onset, median (IQR)                                  N/A   
Age at MRI scan, median (IQR)                                 36 (27-43)   
Epilepsy duration (years), median (IQR)                              N/A   
History of convulsive seizures, n (%)                                N/A   
History of status epilepticus, n (%)                                 N/A   
Epileptogenic lesion on MRI, n (%)                                   N/A   

                                                  Comparison 2  \
                                              p Focal\n(n=203)   
                                                                 
Sex at birth, female/male                <0.001         93/110   
Age of epilepsy onset, median (IQR)         N/A     28 (18-41)   
Age at MRI scan, median (IQR)             0.490     38 (29-51)   
Epilepsy duration (years), median (IQR)     N/A       6 (2-15)   
History of convulsive seizures, n (%)       N/A      155 (76%)   
History of status epilepticus, n (%)        N/A        11 (5%)   
Epileptogenic lesion on MRI, n (%)          N/A       79 (39%)   

                                                                     \
                                        Generalized\n(n=37)       p   
                                                                      
Sex at birth, female/male                             20/17   0.376   
Age of epilepsy onset, median (IQR)               12 (8-18)  <0.001   
Age at MRI scan, median (IQR)                    24 (20-29)  <0.001   
Epilepsy duration (years), median (IQR)           12 (5-19)   0.056   
History of convulsive seizures, n (%)              35 (95%)   0.014   
History of status epilepticus, n (%)                 3 (8%)   0.458   
Epileptogenic lesion on MRI, n (%)                   1 (3%)  <0.001   

                                                    Comparison 3  \
                                        Drug-responsive\n(n=176)   
                                                                   
Sex at birth, female/male                                  82/94   
Age of epilepsy onset, median (IQR)                   26 (18-42)   
Age at MRI scan, median (IQR)                         36 (27-49)   
Epilepsy duration (years), median (IQR)                 5 (1-13)   
History of convulsive seizures, n (%)                  136 (77%)   
History of status epilepticus, n (%)                      7 (4%)   
Epileptogenic lesion on MRI, n (%)                      58 (33%)   

                                                                        
                                        Drug-resistant\n(n=97)       p  
                                                                        
Sex at birth, female/male                                45/52   1.000  
Age of epilepsy onset, median (IQR)                 19 (10-31)  <0.001  
Age at MRI scan, median (IQR)                       36 (26-45)   0.437  
Epilepsy duration (years), median (IQR)              10 (6-23)  <0.001  
History of convulsive seizures, n (%)                 85 (88%)   0.038  
History of status epilepticus, n (%)

Saved: ../outputs/table1_clinical_characteristics.csv
